# Multiclass Chess Style Results

This notebook is intentionally a reporting notebook, not a training notebook. The training logic lives in `scripts/train_multiclass.py` so the same experiment can run locally or on HPC without copying code into notebook cells.

The model predicts which of the 8 players produced a PGN game. It uses a full move sequence, including both the target player's moves and the opponent's moves, because style decisions are easier to interpret when the model also sees the opponent context.

## Method Summary

Each game is converted into a padded tensor sequence. Each timestep is one ply from the game, not only the target player's move.

- The first 34 channels follow the paper-style board representation: before-board pieces, after-board pieces, and board metadata.
- Context channels mark whether the ply was made by the target player or by the opponent.
- Stockfish channels are also board-shaped: scalar evaluation planes plus destination-square heatmaps for the played move and Stockfish's top candidate moves.

This keeps every input as a board-like image, so the CNN sees Stockfish information and board state through the same spatial interface.

In [ ]:
# Imports are kept small because this notebook only reads saved artifacts.
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

# Change this path if you used --output-dir with a different location.
RESULTS_DIR = Path('../results/multiclass_proof')
RESULTS_DIR

## Training Curve

The metrics CSV is written after every epoch, so it is still useful even if an HPC job stops early. Validation accuracy is the main checkpoint-selection signal; test metrics are reported for visibility.

In [ ]:
metrics_path = RESULTS_DIR / 'metrics.csv'

if not metrics_path.exists():
    raise FileNotFoundError(f'Missing {metrics_path}. Run scripts/train_multiclass.py first.')

metrics = pd.read_csv(metrics_path)
metrics

In [ ]:
# Plot accuracy over epochs. A large train/validation gap usually means the
# model is memorizing players' dataset artifacts instead of learning robust style.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(metrics['epoch'], metrics['train_acc'], marker='o', label='train')
ax.plot(metrics['epoch'], metrics['val_acc'], marker='o', label='validation')
ax.plot(metrics['epoch'], metrics['test_acc'], marker='o', label='test')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Multiclass Accuracy')
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Per-Class Metrics

Macro F1 is important here because every player should matter equally. Per-class recall tells us which players are easy or hard for the model to identify.

In [ ]:
report_path = RESULTS_DIR / 'classification_report.csv'
report = pd.read_csv(report_path)
report

In [ ]:
# Keep only real player rows for this plot; sklearn also writes aggregate rows.
player_rows = report[report['label'].isin(['Anand', 'Carlsen', 'Caruana', 'Duda', 'Erigaisi', 'Firouzja', 'Praggnanandhaa', 'Tal'])]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(player_rows['label'], player_rows['f1_score'])
ax.set_xlabel('Player')
ax.set_ylabel('F1 score')
ax.set_title('Per-Player F1')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.25)
plt.show()

## Confusion Matrix

The confusion matrix shows which players the model mixes up. For this project, confusions are often more informative than a single accuracy number because they suggest which styles are similar under the current representation.

In [ ]:
confusion_path = RESULTS_DIR / 'confusion_matrix.csv'
confusion = pd.read_csv(confusion_path, index_col='true_label')
confusion

In [ ]:
# This uses the PNG saved by training if it exists. If not, the DataFrame above
# still contains the full numeric confusion matrix.
image_path = RESULTS_DIR / 'confusion_matrix.png'
if image_path.exists():
    img = plt.imread(image_path)
    plt.figure(figsize=(9, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print(f'Missing {image_path}')

## Run Configuration

The configuration file records the exact command-line choices used by the run. This is useful when comparing local smoke tests against the HPC proof run.

In [ ]:
config_path = RESULTS_DIR / 'run_config.json'
with open(config_path, 'r', encoding='utf-8') as f:
    run_config = json.load(f)

run_config